# Task 1: Netflix Data Cleaning and Preparation
**Goal:** prepare the Netflix dataset for business analytics and reporting.
**Steps:** import, handle missing values, remove duplicates and formatting issues, standardize Country, Rating and Type, export the cleaned file.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv("Dataset.csv")
df = df_raw.copy()          # keep the raw data safe for comparison
print("Shape:", df.shape)
df.head()

Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [4]:
df.describe()

,release_year
count,8790.000000
mean,2014.183163
std,8.825466
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [5]:
print("Missing values:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("\n", df["type"].value_counts())

Missing values:
 show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Exact duplicate rows: 0

 type
Movie      6126
TV Show    2664
Name: count, dtype: int64


In [6]:
placeholders = ["Not Given", "Unknown", "N/A", "NA", "None", ""]

for col in df.columns:
    n = df[col].astype(str).str.strip().isin(placeholders).sum()
    if n > 0:
        print(f"{col}: {n} placeholder values")

title: 1 placeholder values
director: 2588 placeholder values
country: 287 placeholder values


In [7]:
df[["director", "country"]] = df[["director", "country"]].replace("Not Given", np.nan)

print(df[["director", "country"]].isnull().sum())
print((df[["director", "country"]].isnull().mean() * 100).round(1))

director    2588
country      287
dtype: int64
director    29.4
country      3.3
dtype: float64


In [8]:
text_cols = ["show_id", "type", "title", "director", "country",
             "date_added", "rating", "duration", "listed_in"]

for col in text_cols:
    before = df[col].copy()
    df[col] = df[col].str.strip()
    print(f"{col}: {(before.fillna('') != df[col].fillna('')).sum()} values changed")

show_id: 0 values changed
type: 0 values changed
title: 1 values changed
director: 0 values changed
country: 0 values changed
date_added: 0 values changed
rating: 0 values changed
duration: 0 values changed
listed_in: 0 values changed


In [10]:
cols = df.columns.drop("show_id")      # every column except the ID
print("Hidden duplicates:", df.duplicated(subset=cols).sum())
df[df.duplicated(subset=cols, keep=False)].sort_values("title")[["show_id", "title", "release_year"]]

Hidden duplicates: 4


,show_id,title,release_year
2925,s3963,15-Aug,2019
4261,s5967,15-Aug,2019
3285,s4523,22-Jul,2018
4260,s5966,22-Jul,2018
393,s3997,9-Feb,2018
537,s5965,9-Feb,2018
2590,s3372,Consequences,2014
4718,s6530,Consequences,2014


In [11]:
df = df.drop_duplicates(subset=cols, keep="first").reset_index(drop=True)
print("Shape after:", df.shape)

Shape after: (8786, 10)


In [12]:
print(df[["director", "country"]].isnull().sum())

df[["director", "country"]] = df[["director", "country"]].fillna("Unknown")
print("Missing values left:", df.isnull().sum().sum())

director    2587
country      287
dtype: int64
Missing values left: 0


In [13]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%m/%d/%Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()

print(df["date_added"].min(), "to", df["date_added"].max())
df[["date_added", "year_added", "month_added"]].head()

2008-01-01 00:00:00 to 2021-09-25 00:00:00


,date_added,year_added,month_added
0,2021-09-25,2021,September
1,2021-09-24,2021,September
2,2021-09-24,2021,September
3,2021-09-22,2021,September
4,2021-09-24,2021,September


In [14]:
df["duration_value"] = df["duration"].str.extract(r"(\d+)")[0].astype(int)
df["duration_unit"] = df["duration"].str.extract(r"([A-Za-z]+)$")[0].replace({"Seasons": "Season"})

print(df["duration_unit"].value_counts())
print(df.groupby("type")["duration_value"].agg(["min", "max"]))

duration_unit
min       6123
Season    2663
Name: count, dtype: int64
         min  max
type             
Movie      3  312
TV Show    1   17


In [15]:
# Rating: NR and UR both mean "not rated"
df["rating"] = df["rating"].replace({"NR": "Not Rated", "UR": "Not Rated"})
print(df["rating"].value_counts())

# Type: only two valid labels
print("\nTypes:", df["type"].unique())

# Country: no case variants such as "india" and "India"
print("Country case variants:", df["country"].str.lower().nunique() != df["country"].nunique())

rating
TV-MA        3204
TV-14        2155
TV-PG         861
R             798
PG-13         490
TV-Y7         333
TV-Y          306
PG            287
TV-G          220
Not Rated      82
G              41
TV-Y7-FV        6
NC-17           3
Name: count, dtype: int64

Types: ['Movie' 'TV Show']
Country case variants: False


In [16]:
print("Final shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates (ignoring show_id):", df.duplicated(subset=df.columns.drop("show_id")).sum())
print("\n", df["type"].value_counts())
print("\nNew columns:", list(df.columns[10:]))
df.dtypes

Final shape: (8786, 14)
Missing values: 0
Duplicates (ignoring show_id): 0

 type
Movie      6123
TV Show    2663
Name: count, dtype: int64

New columns: ['year_added', 'month_added', 'duration_value', 'duration_unit']


show_id                   object
type                      object
title                     object
director                  object
country                   object
date_added        datetime64[ns]
release_year               int64
rating                    object
duration                  object
listed_in                 object
year_added                 int32
month_added               object
duration_value             int64
duration_unit             object
dtype: object

In [17]:
df.to_csv("Netflix_Cleaned.csv", index=False)
print("Netflix_Cleaned.csv exported successfully!")

Netflix_Cleaned.csv exported successfully!


In [18]:
df_check = pd.read_csv("Netflix_Cleaned.csv", parse_dates=["date_added"])
print("Exported file shape:", df_check.shape)
print("Missing values:", df_check.isnull().sum().sum())
print(df_check["date_added"].dtype)
df_check.head()

Exported file shape: (8786, 14)
Missing values: 0
datetime64[ns]


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,September,90,min
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,September,1,Season
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,September,1,Season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,September,91,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,September,125,min


## Summary of cleaning
| Step | What I did | Result |
|---|---|---|
| Missing values | Found "Not Given" in director (2,588) and country (287); converted to missing, then filled with "Unknown" | 0 missing values; rows kept |
| Formatting | Stripped extra spaces in text columns | 1 title fixed |
| Duplicates | Found 4 hidden duplicates by ignoring show_id | 8,790 rows reduced to 8,786 |
| Dates | Converted date_added to datetime; added year_added and month_added | Ready for trend analysis |
| Duration | Split into duration_value and duration_unit | Movies in minutes, shows in seasons |
| Standardization | Merged NR and UR into "Not Rated"; checked type and country labels | 13 rating categories |

**Final dataset:** 8,786 rows and 14 columns, exported as Netflix_Cleaned.csv.